# EfficientNetV2-L — lesion image classification (MILK supplements)

Reads labels from `dataset/supplements/training_gt.csv` and image keys from `dataset/supplements/training_input.csv`, resolves files under **IMAGES_DIR**, and trains a multi-label head on EfficientNetV2-L.

This notebook is **only** the EfficientNet pipeline (no HiResCAM here).

In [1]:
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# --- paths (override IMAGES_DIR if your files live elsewhere) ---
REPO_ROOT = Path("..").resolve()
SUPP = REPO_ROOT / "dataset" / "supplements"
IMAGES_DIR = REPO_ROOT / "dataset" / "images"  # e.g. dataset/images/ISIC_0081055.jpg

GT_CSV = SUPP / "training_gt.csv"
INPUT_CSV = SUPP / "training_input.csv"

LABEL_COLS = [
    "AKIEC", "BCC", "BEN_OTH", "BKL", "DF", "INF",
    "MAL_OTH", "MEL", "NV", "SCCKA", "VASC",
]

IMG_SIZE = 480  # RX 6750 XT 12GB: keep 480; use 384 only if you need a larger batch
BATCH_SIZE = 16  # 6750 XT: try 16 first, then 20-24; use 8-12 if OOM (DirectML overhead varies)
SEED = 42
AUTOTUNE = tf.data.AUTOTUNE

print("REPO_ROOT:", REPO_ROOT)
print("IMAGES_DIR exists:", IMAGES_DIR.is_dir(), IMAGES_DIR)

REPO_ROOT: E:\RESEARCH_SHIT\MILK
IMAGES_DIR exists: True E:\RESEARCH_SHIT\MILK\dataset\images


In [2]:
def find_image_path(images_dir: Path, isic_id: str) -> Optional[Path]:
    """Resolve ISIC_* to a file in images_dir (.jpg / .jpeg / .png)."""
    for ext in (".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"):
        p = images_dir / f"{isic_id}{ext}"
        if p.is_file():
            return p
    return None


def load_frame() -> pd.DataFrame:
    gt = pd.read_csv(GT_CSV)
    inp = pd.read_csv(INPUT_CSV)
    df = inp.merge(gt, on="lesion_id", how="inner")
    df["image_path"] = df["isic_id"].astype(str).map(lambda i: find_image_path(IMAGES_DIR, i))
    before = len(df)
    df = df.dropna(subset=["image_path"]).reset_index(drop=True)
    print(f"Rows after merge: {before}; with existing image file: {len(df)}")
    return df


df = load_frame()
df[["isic_id", "lesion_id"] + LABEL_COLS].head()

Rows after merge: 10480; with existing image file: 10480


,isic_id,lesion_id,AKIEC,BCC,BEN_OTH,BKL,DF,INF,MAL_OTH,MEL,NV,SCCKA,VASC
0,ISIC_8149219,IL_0000652,0,1,0,0,0,0,0,0,0,0,0
1,ISIC_4671410,IL_0000652,0,1,0,0,0,0,0,0,0,0,0
2,ISIC_3904045,IL_0003176,0,1,0,0,0,0,0,0,0,0,0
3,ISIC_5371928,IL_0003176,0,1,0,0,0,0,0,0,0,0,0
4,ISIC_0791494,IL_0004688,0,1,0,0,0,0,0,0,0,0,0


In [3]:
def decode_image(path, label):
    data = tf.io.read_file(path)
    img = tf.image.decode_image(data, channels=3, expand_animations=False)
    img.set_shape([None, None, 3])
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE], method="bilinear")
    img = keras.applications.efficientnet_v2.preprocess_input(tf.cast(img, tf.float32))
    return img, label


def make_dataset(frame: pd.DataFrame, shuffle: bool):
    paths = frame["image_path"].astype(str).values
    labels = frame[LABEL_COLS].values.astype("float32")
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(min(len(frame), 2048), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(
        lambda p, y: decode_image(p, y),
        num_parallel_calls=AUTOTUNE,
    )
    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds


n = len(df)
if n == 0:
    raise FileNotFoundError(
        f"No rows with images under {IMAGES_DIR}. "
        "Place ISIC_* image files there or set IMAGES_DIR."
    )

idx = np.arange(n)
rng = np.random.default_rng(SEED)
rng.shuffle(idx)
split = int(0.85 * n)
train_idx, val_idx = idx[:split], idx[split:]
train_df = df.iloc[train_idx].reset_index(drop=True)
val_df = df.iloc[val_idx].reset_index(drop=True)

train_ds = make_dataset(train_df, shuffle=True)
val_ds = make_dataset(val_df, shuffle=False)
len(train_df), len(val_df)

(8908, 1572)

In [4]:
def build_model(num_labels: int) -> keras.Model:
    # pooling=None + GAP keeps the same weight layout as `hires_cam.ipynb` (spatial maps).
    base = keras.applications.EfficientNetV2L(
        include_top=False,
        weights="imagenet",
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        pooling=None,
    )
    base.trainable = False  # unfreeze later for fine-tune if you want
    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = base(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(num_labels, activation="sigmoid")(x)
    return keras.Model(inputs, outputs, name="efficientnetv2l_multilabel")


model = build_model(len(LABEL_COLS))
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss=keras.losses.BinaryCrossentropy(),
    metrics=[keras.metrics.AUC(name="auc", multi_label=True)],
)
model.summary()

Model: "efficientnetv2l_multilabel"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_2 (InputLayer)        [(None, 480, 480, 3)]     0         
                                                                 
 efficientnetv2-l (Functiona  (None, 15, 15, 1280)     117746848 
 l)                                                              
                                                                 
 global_average_pooling2d (G  (None, 1280)             0         
 lobalAveragePooling2D)                                          
                                                                 
 dropout (Dropout)           (None, 1280)              0         
                                                                 
 dense (Dense)               (None, 11)                14091     
                                                                 
Total params: 117,760,939
Trainable para

In [5]:
EPOCHS_HEAD = 20  # increase when ready

callbacks = [
    keras.callbacks.ModelCheckpoint(
        "efficientnetv2l_best.weights.h5",
        monitor="val_loss",
        save_best_only=True,
        save_weights_only=True,
    ),
]

history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_HEAD, callbacks=callbacks)
history.history.keys()

Epoch 1/20
557/557 [==============================] - 370s 644ms/step - loss: 0.3135 - auc: 0.5348 - val_loss: 0.2253 - val_auc: 0.5989
Epoch 2/20
557/557 [==============================] - 355s 637ms/step - loss: 0.2146 - auc: 0.6124 - val_loss: 0.2072 - val_auc: 0.6373
Epoch 3/20
557/557 [==============================] - 354s 636ms/step - loss: 0.2024 - auc: 0.6576 - val_loss: 0.2001 - val_auc: 0.6642
Epoch 4/20
557/557 [==============================] - 353s 634ms/step - loss: 0.1960 - auc: 0.6634 - val_loss: 0.1957 - val_auc: 0.6838
Epoch 5/20
557/557 [==============================] - 352s 633ms/step - loss: 0.1917 - auc: 0.6836 - val_loss: 0.1925 - val_auc: 0.7077
Epoch 6/20
557/557 [==============================] - 353s 634ms/step - loss: 0.1888 - auc: 0.7028 - val_loss: 0.1901 - val_auc: 0.7073
Epoch 7/20
557/557 [==============================] - 353s 634ms/step - loss: 0.1863 - auc: 0.7152 - val_loss: 0.1882 - val_auc: 0.7159
Epoch 8/20
557/557 [============================

dict_keys(['loss', 'auc', 'val_loss', 'val_auc'])

In [6]:
# Stage 2: fine-tune upper EfficientNetV2-L layers (after head training)
# Run this after the previous `model.fit(...)` cell finishes.

# 1) Unfreeze only top part of backbone for stable fine-tuning.
base = model.get_layer("efficientnetv2-l")
base.trainable = True

# Keep early layers frozen; unfreeze only the last ~20%% layers.
cut = int(len(base.layers) * 0.8)
for layer in base.layers[:cut]:
    layer.trainable = False

# 2) Re-compile with a smaller learning rate.
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss=keras.losses.BinaryCrossentropy(),
    metrics=[keras.metrics.AUC(name="auc", multi_label=True)],
)

# 3) Fine-tune with LR schedule + early stopping.
FINE_TUNE_EPOCHS = 12

fine_tune_callbacks = [
    keras.callbacks.ModelCheckpoint(
        "efficientnetv2l_best_ft.weights.h5",
        monitor="val_loss",
        save_best_only=True,
        save_weights_only=True,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-7,
        verbose=1,
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True,
        verbose=1,
    ),
]

history_ft = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=FINE_TUNE_EPOCHS,
    callbacks=fine_tune_callbacks,
)

print("Trainable variables:", len(model.trainable_variables))

Epoch 1/12
557/557 [==============================] - 598s 1s/step - loss: 0.1555 - auc: 0.8223 - val_loss: 0.1481 - val_auc: 0.8392 - lr: 1.0000e-05
Epoch 2/12
557/557 [==============================] - 579s 1s/step - loss: 0.1296 - auc: 0.8906 - val_loss: 0.1423 - val_auc: 0.8664 - lr: 1.0000e-05
Epoch 3/12
557/557 [==============================] - 579s 1s/step - loss: 0.1051 - auc: 0.9210 - val_loss: 0.1405 - val_auc: 0.8556 - lr: 1.0000e-05
Epoch 4/12
557/557 [==============================] - 578s 1s/step - loss: 0.0765 - auc: 0.9578 - val_loss: 0.1545 - val_auc: 0.8487 - lr: 1.0000e-05
Epoch 5/12
557/557 [==============================] - ETA: 0s - loss: 0.0474 - auc: 0.9802
Epoch 5: ReduceLROnPlateau reducing learning rate to 4.999999873689376e-06.
557/557 [==============================] - 578s 1s/step - loss: 0.0474 - auc: 0.9802 - val_loss: 0.1869 - val_auc: 0.8136 - lr: 1.0000e-05
Epoch 6/12
557/557 [==============================] - 578s 1s/step - loss: 0.0215 - auc: 0.993

In [7]:
# Stage 3 prep: stronger augmentation pipeline (train only)

def decode_image_with_aug(path, label, training: bool):
    data = tf.io.read_file(path)
    img = tf.image.decode_image(data, channels=3, expand_animations=False)
    img.set_shape([None, None, 3])
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE], method="bilinear")
    img = tf.cast(img, tf.float32) / 255.0

    if training:
        img = tf.image.random_flip_left_right(img)
        img = tf.image.random_flip_up_down(img)
        img = tf.image.random_brightness(img, max_delta=0.12)
        img = tf.image.random_contrast(img, lower=0.85, upper=1.15)
        img = tf.image.random_saturation(img, lower=0.85, upper=1.15)

    img = tf.clip_by_value(img, 0.0, 1.0)
    img = keras.applications.efficientnet_v2.preprocess_input(img * 255.0)
    return img, label


def make_dataset_aug(frame: pd.DataFrame, training: bool):
    paths = frame["image_path"].astype(str).values
    labels = frame[LABEL_COLS].values.astype("float32")
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        ds = ds.shuffle(min(len(frame), 2048), seed=SEED, reshuffle_each_iteration=True)

    ds = ds.map(
        lambda p, y: decode_image_with_aug(p, y, training=training),
        num_parallel_calls=AUTOTUNE,
    )
    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds


train_ds_aug = make_dataset_aug(train_df, training=True)
val_ds_plain = make_dataset_aug(val_df, training=False)
print("Augmented train/val datasets ready")

Augmented train/val datasets ready


In [8]:
# Stage 3 training: focal loss + gentler fine-tuning depth
# Start from best stage-2 checkpoint before trying new strategy.

model.load_weights("efficientnetv2l_best_ft.weights.h5")

base = model.get_layer("efficientnetv2-l")
base.trainable = True

# Unfreeze only last 10% of backbone to reduce overfitting.
cut = int(len(base.layers) * 0.9)
for layer in base.layers[:cut]:
    layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=5e-6),
    loss=keras.losses.BinaryFocalCrossentropy(gamma=2.0),
    metrics=[keras.metrics.AUC(name="auc", multi_label=True)],
)

FINE_TUNE_EPOCHS_2 = 8

stage3_callbacks = [
    keras.callbacks.ModelCheckpoint(
        "efficientnetv2l_best_stage3.weights.h5",
        monitor="val_loss",
        save_best_only=True,
        save_weights_only=True,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-7,
        verbose=1,
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True,
        verbose=1,
    ),
]

history_stage3 = model.fit(
    train_ds_aug,
    validation_data=val_ds_plain,
    epochs=FINE_TUNE_EPOCHS_2,
    callbacks=stage3_callbacks,
)

print("Trainable variables (stage 3):", len(model.trainable_variables))

Epoch 1/8
557/557 [==============================] - 524s 919ms/step - loss: 0.0260 - auc: 0.9476 - val_loss: 0.0464 - val_auc: 0.8972 - lr: 5.0000e-06
Epoch 2/8
557/557 [==============================] - 505s 907ms/step - loss: 0.0210 - auc: 0.9730 - val_loss: 0.0464 - val_auc: 0.8943 - lr: 5.0000e-06
Epoch 3/8
557/557 [==============================] - ETA: 0s - loss: 0.0183 - auc: 0.9808
Epoch 3: ReduceLROnPlateau reducing learning rate to 2.499999936844688e-06.
557/557 [==============================] - 504s 906ms/step - loss: 0.0183 - auc: 0.9808 - val_loss: 0.0511 - val_auc: 0.8907 - lr: 5.0000e-06
Epoch 4/8
557/557 [==============================] - 502s 902ms/step - loss: 0.0154 - auc: 0.9861 - val_loss: 0.0522 - val_auc: 0.8950 - lr: 2.5000e-06
Epoch 4: early stopping
Trainable variables (stage 3): 93


In [9]:
# Threshold tuning on validation set (maximize per-class F1)

model.load_weights("efficientnetv2l_best_stage3.weights.h5")

# Collect labels/predictions from validation set.
val_y_true = np.concatenate([y.numpy() for _, y in val_ds_plain], axis=0)
val_y_pred = model.predict(val_ds_plain, verbose=1)


def f1_at_threshold(y_true_col, y_pred_col, thr: float, eps: float = 1e-9):
    y_hat = (y_pred_col >= thr).astype(np.float32)
    tp = np.sum(y_hat * y_true_col)
    fp = np.sum(y_hat * (1.0 - y_true_col))
    fn = np.sum((1.0 - y_hat) * y_true_col)
    precision = tp / (tp + fp + eps)
    recall = tp / (tp + fn + eps)
    return (2.0 * precision * recall) / (precision + recall + eps)


threshold_grid = np.linspace(0.05, 0.95, 19)
best_thresholds = []
best_f1s = []

for i, cls in enumerate(LABEL_COLS):
    y_t = val_y_true[:, i]
    y_p = val_y_pred[:, i]
    f1_scores = [f1_at_threshold(y_t, y_p, t) for t in threshold_grid]
    best_idx = int(np.argmax(f1_scores))
    best_thresholds.append(float(threshold_grid[best_idx]))
    best_f1s.append(float(f1_scores[best_idx]))

threshold_df = pd.DataFrame(
    {
        "class": LABEL_COLS,
        "best_threshold": best_thresholds,
        "best_f1": best_f1s,
    }
).sort_values("best_f1", ascending=False)

print(threshold_df)
print("Macro-F1 (tuned):", float(np.mean(best_f1s)))

np.save("efficientnetv2l_stage3_thresholds.npy", np.array(best_thresholds, dtype=np.float32))
model.save("efficientnetv2l_multilabel_stage3_best.keras")
print("Saved thresholds to efficientnetv2l_stage3_thresholds.npy")
print("Saved model to efficientnetv2l_multilabel_stage3_best.keras")

99/99 [==============================] - 56s 539ms/step
      class  best_threshold   best_f1
1       BCC            0.50  0.843646
8        NV            0.40  0.724070
9     SCCKA            0.40  0.619355
10     VASC            0.40  0.571429
7       MEL            0.35  0.502857
3       BKL            0.40  0.417344
0     AKIEC            0.40  0.376963
4        DF            0.40  0.357143
2   BEN_OTH            0.45  0.250000
5       INF            0.45  0.230769
6   MAL_OTH            0.05  0.031496
Macro-F1 (tuned): 0.44773388312343365
Saved thresholds to efficientnetv2l_stage3_thresholds.npy
Saved model to efficientnetv2l_multilabel_stage3_best.keras


In [10]:
# Final export cell: save deploy-ready model artifacts

FINAL_MODEL_PATH = "efficientnetv2l_multilabel_final.keras"
FINAL_WEIGHTS_PATH = "efficientnetv2l_multilabel_final.weights.h5"
FINAL_THRESH_PATH = "efficientnetv2l_multilabel_final_thresholds.npy"

# Ensure we export the best stage-3 checkpoint.
model.load_weights("efficientnetv2l_best_stage3.weights.h5")

# Save full model + weights.
model.save(FINAL_MODEL_PATH)
model.save_weights(FINAL_WEIGHTS_PATH)

# Save thresholds (expects `best_thresholds` from threshold-tuning cell).
if "best_thresholds" in globals():
    np.save(FINAL_THRESH_PATH, np.array(best_thresholds, dtype=np.float32))
else:
    # Fallback: copy existing saved thresholds if already generated.
    try:
        t = np.load("efficientnetv2l_stage3_thresholds.npy")
        np.save(FINAL_THRESH_PATH, t.astype(np.float32))
    except Exception as e:
        print("Thresholds not found. Run threshold tuning cell first.")
        print("Details:", e)

print("Saved:")
print("-", FINAL_MODEL_PATH)
print("-", FINAL_WEIGHTS_PATH)
print("-", FINAL_THRESH_PATH)

Saved:
- efficientnetv2l_multilabel_final.keras
- efficientnetv2l_multilabel_final.weights.h5
- efficientnetv2l_multilabel_final_thresholds.npy


In [11]:
# Safer split: group by lesion_id to reduce leakage across train/val/test
# Run this cell before re-training if you want more trustworthy metrics.

GROUP_TEST_SIZE = 0.15
GROUP_VAL_SIZE = 0.15

lesion_ids = df["lesion_id"].astype(str).unique()
rng = np.random.default_rng(SEED)
rng.shuffle(lesion_ids)

n_lesions = len(lesion_ids)
n_test = int(round(n_lesions * GROUP_TEST_SIZE))
n_val = int(round(n_lesions * GROUP_VAL_SIZE))

# Split unique lesion groups, then map back to image rows.
test_lesions = set(lesion_ids[:n_test])
val_lesions = set(lesion_ids[n_test:n_test + n_val])
train_lesions = set(lesion_ids[n_test + n_val:])

train_df_grouped = df[df["lesion_id"].astype(str).isin(train_lesions)].reset_index(drop=True)
val_df_grouped = df[df["lesion_id"].astype(str).isin(val_lesions)].reset_index(drop=True)
test_df_grouped = df[df["lesion_id"].astype(str).isin(test_lesions)].reset_index(drop=True)

print("Grouped split sizes (rows):", len(train_df_grouped), len(val_df_grouped), len(test_df_grouped))
print("Grouped split sizes (lesions):", len(train_lesions), len(val_lesions), len(test_lesions))
print(
    "Overlap check:",
    len(train_lesions & val_lesions),
    len(train_lesions & test_lesions),
    len(val_lesions & test_lesions),
)

train_ds_grouped = make_dataset(train_df_grouped, shuffle=True)
val_ds_grouped = make_dataset(val_df_grouped, shuffle=False)
test_ds_grouped = make_dataset(test_df_grouped, shuffle=False)

Grouped split sizes (rows): 7336 1572 1572
Grouped split sizes (lesions): 3668 786 786
Overlap check: 0 0 0


In [12]:
# Test-set evaluation with saved model + tuned thresholds
# Run after re-training on the grouped split above.

FINAL_MODEL_PATH = "efficientnetv2l_multilabel_final.keras"
FINAL_THRESH_PATH = "efficientnetv2l_multilabel_final_thresholds.npy"

final_model = keras.models.load_model(FINAL_MODEL_PATH)
final_thresholds = np.load(FINAL_THRESH_PATH)

# Scalar metric from Keras
print(final_model.evaluate(test_ds_grouped, return_dict=True, verbose=1))

# Threshold-aware multilabel evaluation on held-out test data

test_y_true = np.concatenate([y.numpy() for _, y in test_ds_grouped], axis=0)
test_y_pred = final_model.predict(test_ds_grouped, verbose=1)
test_y_hat = (test_y_pred >= final_thresholds.reshape(1, -1)).astype(np.float32)


def f1_score_binary(y_true_col, y_hat_col, eps: float = 1e-9):
    tp = np.sum(y_hat_col * y_true_col)
    fp = np.sum(y_hat_col * (1.0 - y_true_col))
    fn = np.sum((1.0 - y_hat_col) * y_true_col)
    precision = tp / (tp + fp + eps)
    recall = tp / (tp + fn + eps)
    return (2.0 * precision * recall) / (precision + recall + eps)


per_class_test_f1 = []
for i, cls in enumerate(LABEL_COLS):
    cls_f1 = f1_score_binary(test_y_true[:, i], test_y_hat[:, i])
    per_class_test_f1.append((cls, float(cls_f1)))

print(pd.DataFrame(per_class_test_f1, columns=["class", "test_f1"]).sort_values("test_f1", ascending=False))
print("Macro-F1 (test, tuned):", float(np.mean([x[1] for x in per_class_test_f1])))

99/99 [==============================] - 59s 541ms/step - loss: 0.0180 - auc: 0.8918
{'loss': 0.017983291298151016, 'auc': 0.8918094635009766}
99/99 [==============================] - 56s 538ms/step
      class   test_f1
10     VASC  0.960000
1       BCC  0.937581
8        NV  0.886957
9     SCCKA  0.861789
3       BKL  0.739496
2   BEN_OTH  0.717949
7       MEL  0.702875
4        DF  0.608696
0     AKIEC  0.577320
5       INF  0.500000
6   MAL_OTH  0.000000
Macro-F1 (test, tuned): 0.6811510513039702


In [4]:
# V1 thesis export (fully standalone: run this cell only)
from pathlib import Path
from typing import Optional
import json

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras

# ---- constants ----
LABEL_COLS = [
    "AKIEC", "BCC", "BEN_OTH", "BKL", "DF", "INF",
    "MAL_OTH", "MEL", "NV", "SCCKA", "VASC",
]

IMG_SIZE = 480
BATCH_SIZE = 16
SEED = 42
AUTOTUNE = tf.data.AUTOTUNE

REPO_ROOT = Path("..").resolve()
SUPP = REPO_ROOT / "dataset" / "supplements"
IMAGES_DIR = REPO_ROOT / "dataset" / "images"
GT_CSV = SUPP / "training_gt.csv"
INPUT_CSV = SUPP / "training_input.csv"

MODEL_PATH = "efficientnetv2l_multilabel_final.keras"
THRESH_PATH = "efficientnetv2l_multilabel_final_thresholds.npy"

V1_OUT_DIR = Path("thesis_v1_results")
V1_OUT_DIR.mkdir(exist_ok=True)


# ---- helpers ----
def find_image_path(images_dir: Path, isic_id: str) -> Optional[Path]:
    for ext in (".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"):
        p = images_dir / f"{isic_id}{ext}"
        if p.is_file():
            return p
    return None


def decode_image(path, label):
    data = tf.io.read_file(path)
    img = tf.image.decode_image(data, channels=3, expand_animations=False)
    img.set_shape([None, None, 3])
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE], method="bilinear")
    img = keras.applications.efficientnet_v2.preprocess_input(tf.cast(img, tf.float32))
    return img, label


def make_dataset(frame: pd.DataFrame, shuffle: bool = False):
    paths = frame["image_path"].astype(str).values
    labels = frame[LABEL_COLS].values.astype("float32")
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(min(len(frame), 2048), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(lambda p, y: decode_image(p, y), num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds


def f1_score_binary(y_true_col, y_hat_col, eps: float = 1e-9):
    tp = np.sum(y_hat_col * y_true_col)
    fp = np.sum(y_hat_col * (1.0 - y_true_col))
    fn = np.sum((1.0 - y_hat_col) * y_true_col)
    precision = tp / (tp + fp + eps)
    recall = tp / (tp + fn + eps)
    return (2.0 * precision * recall) / (precision + recall + eps)


# ---- rebuild grouped test split ----
gt = pd.read_csv(GT_CSV)
inp = pd.read_csv(INPUT_CSV)
df = inp.merge(gt, on="lesion_id", how="inner")
df["image_path"] = df["isic_id"].astype(str).map(lambda i: find_image_path(IMAGES_DIR, i))
df = df.dropna(subset=["image_path"]).reset_index(drop=True)

lesion_ids = df["lesion_id"].astype(str).unique()
rng = np.random.default_rng(SEED)
rng.shuffle(lesion_ids)

n_lesions = len(lesion_ids)
n_test = int(round(n_lesions * 0.15))

test_lesions = set(lesion_ids[:n_test])
test_df_grouped = df[df["lesion_id"].astype(str).isin(test_lesions)].reset_index(drop=True)
test_ds_grouped = make_dataset(test_df_grouped, shuffle=False)

# ---- load model + thresholds ----
v1_model = keras.models.load_model(MODEL_PATH)
v1_thresholds = np.load(THRESH_PATH)

metrics = v1_model.evaluate(test_ds_grouped, return_dict=True, verbose=1)

test_y_true = np.concatenate([y.numpy() for _, y in test_ds_grouped], axis=0)
test_y_prob = v1_model.predict(test_ds_grouped, verbose=1)
test_y_hat = (test_y_prob >= v1_thresholds.reshape(1, -1)).astype(np.float32)

rows = []
for i, cls in enumerate(LABEL_COLS):
    support = int(np.sum(test_y_true[:, i]))
    f1 = float(f1_score_binary(test_y_true[:, i], test_y_hat[:, i]))
    rows.append({
        "class": cls,
        "threshold": float(v1_thresholds[i]),
        "support_test": support,
        "f1_test": f1,
    })

per_class_df = pd.DataFrame(rows).sort_values("f1_test", ascending=False)
macro_f1 = float(per_class_df["f1_test"].mean())

# ---- save thesis bundle ----
per_class_path = V1_OUT_DIR / "per_class_test_metrics.csv"
probs_path = V1_OUT_DIR / "test_probabilities.csv"
summary_path = V1_OUT_DIR / "summary_metrics.json"

per_class_df.to_csv(per_class_path, index=False)

test_probs_df = pd.DataFrame(test_y_prob, columns=[f"prob_{c}" for c in LABEL_COLS])
for i, c in enumerate(LABEL_COLS):
    test_probs_df[f"pred_{c}"] = test_y_hat[:, i].astype(int)
    test_probs_df[f"true_{c}"] = test_y_true[:, i].astype(int)

test_probs_df.to_csv(probs_path, index=False)

summary = {
    "model": MODEL_PATH,
    "thresholds": THRESH_PATH,
    "test_loss": float(metrics.get("loss", np.nan)),
    "test_auc": float(metrics.get("auc", np.nan)),
    "test_macro_f1_tuned": macro_f1,
    "n_test_samples": int(test_y_true.shape[0]),
}
summary_path.write_text(json.dumps(summary, indent=2))

print("Saved V1 thesis bundle to:", V1_OUT_DIR.resolve())
print("-", per_class_path.name)
print("-", probs_path.name)
print("-", summary_path.name)
print("\nSummary:")
print(summary)
per_class_df

RuntimeError: Missing required variables: LABEL_COLS, test_ds_grouped. Run setup + grouped split cells first.